# Project - Airline Flight Booking Assistant

In [73]:
# import necessary libraries
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [54]:
# Load environment variables from .env file

load_dotenv(override=True)

anthropic_base_url = os.getenv("anthropic_url")       # was: anthropic_base_url
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")    # was: anthropic_api_key (lowercase)
MODEL = "claude-sonnet-4-6"                           # was: claude-sonnet-4-5-20250929
openai = OpenAI(base_url=anthropic_base_url, api_key=anthropic_api_key)

DB = os.path.join(os.path.dirname(os.path.abspath("day5.ipynb")), "prices.db")

In [74]:
# function to set ticket price in the database
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO ticket_prices (destination_city, price) VALUES (?, ?) ON CONFLICT(destination_city) DO UPDATE SET price = ?', (city.lower(), price, price))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [75]:
# calling set_ticket_price for each city and price in the dictionary
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999, "new york": 1200, "berlin": 750, "rome": 850, "dubai": 1100, "singapore": 1300, "hong kong": 1250, "mumbai": 600, "cairo": 550, "moscow": 700, "rio de janeiro": 950, "cape town": 800, "bangkok": 650, "istanbul": 900, "seoul": 1150, "toronto": 1050, "san francisco": 1400, "barcelona": 850, "amsterdam": 800, "vienna": 750, "prague": 700, "lisbon": 650, "budapest": 600, "warsaw": 550, "athens": 500, "copenhagen": 450, "stockholm": 400, "helsinki": 350, "oslo": 300, "zurich": 250, "brussels": 200, "dublin": 150}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [76]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [77]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Get ticket price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM ticket_prices WHERE destination_city = ?', (city.lower(),))
        result = cursor.fetchone()
        if result:
            return f"The ticket price to {city} is ${result[0]}."
        else:
            return f"Sorry, no price data available for {city}."

In [78]:
get_ticket_price("mumbai")

DATABASE TOOL CALLED: Get ticket price for mumbai


'The ticket price to mumbai is $600.'

In [84]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [68]:
# Chat function without Tool calls 

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


In [85]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [86]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages", title="FlightAI Ticket Price Assistant").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Get ticket price for London
DATABASE TOOL CALLED: Get ticket price for Mumbai
